![Universidad Iberoamericana del Ecuador](https://raw.githubusercontent.com/andresrubiop/ml-unibe-estudiantes/main/utils/logo-unibe-color.png)

<div style="background:#004890;color:#FFFFFF;padding:14px 18px;border-radius:10px;margin:6px 0 12px 0;border-bottom:4px solid #FCC000"><div style="font-size:12px;letter-spacing:.08em;text-transform:uppercase;opacity:.9">Machine Learning · Sesión 2 · Notebook del estudiante · no calificable</div><div style="font-size:22px;font-weight:700;margin-top:4px">L2.3 · Extraer conocimiento: selección de variables, evolución, reglas y elección del modelo</div><div style="font-size:12px;opacity:.9;margin-top:4px">Posgrado · Maestría en Big Data y Ciencia de Datos · UNIB.E</div></div>

| | |
|---|---|
| **Objetivo** | Pasar de «un modelo que predice» a **conocimiento** que una persona puede revisar: qué variables importan, qué reglas explican una decisión y qué modelo elegir con argumentos. |
| **Duración** | ≈ 90 min (base de la **Clase 3**, actividad integradora) |
| **Teoría** | Manual M2 §1–§5 · Animaciones A2.9 a A2.11 y A2.14 a A2.16 |
| **Datos icónicos** | **Titanic** (891 pasajeros), el problema de la **propina** de la lógica difusa |
| **Escala realista** | **cáncer de mama de Wisconsin** (569 tumores, 30 medidas; Street, Wolberg y Mangasarian, 1993) con 20 variables de ruido añadidas |

**Partes:** 1 · selección de variables: filtro, envoltura e incrustada → 2 · un algoritmo genético que elige variables → 3 · reglas difusas de Mamdani → 4 · un sistema experto y las reglas que aprende un árbol → 5 · selección del modelo con validación cruzada anidada y calibración.

## 0 · Configuración

In [ ]:
import os
import time
import urllib.request
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from cycler import cycler

warnings.filterwarnings("ignore", category=FutureWarning)
SEED = 2026
UNIBE = {"azul": "#004890", "naranja": "#C2410C", "dorado": "#A87900", "verde": "#3A7D44", "gris": "#77787B"}
plt.rcParams.update({
    "axes.prop_cycle": cycler(color=list(UNIBE.values())), "axes.titlecolor": UNIBE["azul"],
    "axes.titleweight": "bold", "axes.edgecolor": UNIBE["gris"], "axes.grid": True, "grid.color": "#E8EDF2",
    "axes.spines.top": False, "axes.spines.right": False, "figure.dpi": 110, "legend.frameon": False,
})
CACHE = os.path.expanduser("~/.cache/ml-unibe")


def cargar_titanic():
    """Titanic desde el repositorio público de seaborn-data (una sola descarga)."""
    os.makedirs(CACHE, exist_ok=True)
    ruta = os.path.join(CACHE, "titanic.csv")
    if not os.path.exists(ruta):
        urllib.request.urlretrieve("https://raw.githubusercontent.com/mwaskom/seaborn-data/master/titanic.csv", ruta)
    return pd.read_csv(ruta)


print(f"scikit-learn {sklearn.__version__} · semilla {SEED}")

## Parte 1 · Selección de variables

Más variables no siempre es mejor: las irrelevantes agregan ruido, cuestan dinero (cada examen médico se paga) y hacen el modelo más difícil de explicar. Tres familias de métodos:

| Familia | Idea | Ejemplo en scikit-learn |
|---|---|---|
| **Filtro** | puntúa cada variable sola, antes de modelar | `SelectKBest(f_classif)` |
| **Envoltura** (*wrapper*) | prueba subconjuntos con el modelo y se queda con el mejor | `RFE`, `SequentialFeatureSelector`, algoritmos genéticos |
| **Incrustada** (*embedded*) | el propio entrenamiento elimina variables | Lasso / logística con penalización L1, importancias de un bosque |

Usamos el conjunto de **cáncer de mama**: 30 medidas de núcleos celulares para decidir si un tumor es maligno o benigno. Le añadimos **20 columnas de ruido puro**, para saber de antemano qué variables son inútiles y ver qué método las descarta.

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split

bc = load_breast_cancer()
rng = np.random.default_rng(SEED)
ruido = rng.normal(size=(len(bc.data), 20))
X_bc = np.hstack([bc.data, ruido])
nombres = list(bc.feature_names) + [f"ruido {k + 1}" for k in range(20)]
es_ruido = np.r_[np.zeros(30), np.ones(20)].astype(bool)
y_bc = 1 - bc.target                                   # 1 = maligno (la clase que nos interesa detectar)
X_tr, X_te, y_tr, y_te = train_test_split(X_bc, y_bc, test_size=0.25, stratify=y_bc, random_state=SEED)
CV = StratifiedKFold(5, shuffle=True, random_state=SEED)
print(f"Entrenamiento {X_tr.shape} · prueba {X_te.shape} · malignos {y_bc.mean():.0%}")

**Tu turno.** Aplica los tres métodos para quedarte con **10 variables** y cuenta cuántas de ruido se cuelan en cada uno.

- Filtro: `SelectKBest(f_classif, k=10)`.
- Envoltura: `RFE(LogisticRegression(max_iter=2000), n_features_to_select=10)`, que elimina la variable menos útil una a una.
- Incrustada: `LogisticRegression(l1_ratio=1.0, solver="liblinear", C=0.05)` (`l1_ratio=1` es la penalización L1 pura); las variables con coeficiente distinto de cero son las elegidas.

Todo dentro de un pipeline con `StandardScaler`.

In [ ]:
from sklearn.feature_selection import RFE, SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# TODO: obtén tres máscaras booleanas de 50 elementos (True = variable elegida): m_filtro, m_rfe, m_l1.
m_filtro = ...
m_rfe = ...
m_l1 = ...
filas = []
for nombre, m in [("todas", np.ones(50, bool)), ("filtro (ANOVA F)", m_filtro), ("envoltura (RFE)", m_rfe), ("incrustada (L1)", m_l1)]:
    acc = cross_val_score(make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)), X_tr[:, m], y_tr, cv=CV).mean()
    filas.append({"método": nombre, "variables": int(m.sum()), "de ruido": int((m & es_ruido).sum()), "exactitud VC": acc})
print(pd.DataFrame(filas).round(3).to_string(index=False))
print("\nElegidas por el filtro:", ", ".join(np.array(nombres)[m_filtro]))
print("Elegidas por L1:", ", ".join(np.array(nombres)[m_l1]))

Ningún método eligió ruido. RFE y L1 igualan o superan a las 50 variables con 10 o menos. El filtro queda algo por debajo: mira cada variable **por separado** y elige radio, perímetro y área, en su versión media y en la peor, que dicen casi lo mismo. Fíjate también en que los métodos eligen variables distintas: medidas tan correlacionadas son **intercambiables**, y «la variable elegida» no siempre es «la única que importa».

### Importancia por permutación

Otra forma de preguntar «¿qué importa?» a un modelo ya entrenado: **desordenar** una columna en la prueba y medir cuánto empeora. Si no empeora, el modelo no la usaba.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance

bosque = RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=-1).fit(X_tr, y_tr)
imp = permutation_importance(bosque, X_te, y_te, n_repeats=20, random_state=SEED, n_jobs=-1)
orden = np.argsort(imp.importances_mean)[::-1][:12]
plt.figure(figsize=(7, 3.6))
plt.barh([nombres[i] for i in orden][::-1], imp.importances_mean[orden][::-1],
         xerr=imp.importances_std[orden][::-1], color=[UNIBE["naranja"] if es_ruido[i] else UNIBE["azul"] for i in orden][::-1])
plt.title("Importancia por permutación (bosque aleatorio, prueba)"); plt.xlabel("caída de exactitud"); plt.show()

## Parte 2 · Un algoritmo genético que elige variables

Con 50 variables hay \(2^{50} \approx 10^{15}\) subconjuntos: imposible probarlos todos. Un **algoritmo genético** (Holland, 1975; Goldberg, 1989) imita la evolución (A2.14):

1. **Población:** cromosomas de 50 bits; 1 = la variable entra.
2. **Aptitud:** exactitud de validación cruzada con esas variables, menos una penalización por cada variable.
3. **Selección por torneo**, **cruce** de un punto y **mutación** bit a bit; los 2 mejores pasan intactos (**elitismo**).

Lo programamos completo con NumPy. La única pieza que te toca es la mutación.

In [ ]:
from sklearn.model_selection import cross_val_score

cache = {}


CV_GA = StratifiedKFold(5, shuffle=True, random_state=0)   # pliegues fijos: todos los cromosomas se comparan igual


def aptitud(crom, penal=0.005):
    """Exactitud media de VC (5 pliegues) de una logística con las variables marcadas, menos una penalización."""
    clave = crom.tobytes()
    if clave not in cache:
        if crom.sum() == 0:
            cache[clave] = 0.0
        else:
            modelo = make_pipeline(StandardScaler(), LogisticRegression(max_iter=500))
            cache[clave] = cross_val_score(modelo, X_tr[:, crom.astype(bool)], y_tr, cv=CV_GA).mean()
    return cache[clave] - penal * crom.sum()


def torneo(pob, apt, rng, k=3):
    """Elige k individuos al azar y devuelve el más apto."""
    ids = rng.integers(0, len(pob), k)
    return pob[ids[np.argmax(apt[ids])]]


def cruce(a, b, rng):
    """Cruce de un punto: el comienzo de un padre y el final del otro."""
    corte = rng.integers(1, len(a))
    return np.concatenate([a[:corte], b[corte:]])

**Tu turno.** Escribe `mutar`: cada bit cambia (0 → 1 o 1 → 0) con probabilidad `p`. Pista: `rng.random(len(crom)) < p` te da qué bits cambian.

In [ ]:
# TODO: devuelve una copia del cromosoma con los bits mutados.
def mutar(crom, rng, p=0.02):
    ...


prueba = mutar(np.zeros(50, dtype=np.int8), np.random.default_rng(0), p=0.5)
assert prueba.dtype == np.int8 and 10 < prueba.sum() < 40, "Con p = 0,5 cambia más o menos la mitad de los bits."

In [ ]:
def evolucionar(generaciones=25, n=30, p_mut=0.02, semilla=SEED):
    rng = np.random.default_rng(semilla)
    pob = (rng.random((n, 50)) < 0.3).astype(np.int8)
    hist = []
    for g in range(generaciones):
        apt = np.array([aptitud(c) for c in pob])
        orden = np.argsort(apt)[::-1]
        pob, apt = pob[orden], apt[orden]
        hist.append((apt[0], apt.mean(), pob.mean(axis=0)))
        hijos = [pob[0].copy(), pob[1].copy()]                       # elitismo
        while len(hijos) < n:
            hijos.append(mutar(cruce(torneo(pob, apt, rng), torneo(pob, apt, rng), rng), rng, p_mut))
        pob = np.array(hijos)
    apt = np.array([aptitud(c) for c in pob])
    return pob[np.argmax(apt)], hist


t0 = time.perf_counter()
mejor, hist = evolucionar()
print(f"{time.perf_counter() - t0:.0f} s · {len(cache)} subconjuntos evaluados de 10^15 posibles")
fig, axs = plt.subplots(1, 2, figsize=(12, 3.4), gridspec_kw={"width_ratios": [1, 2]})
axs[0].plot([h[0] for h in hist], color=UNIBE["verde"], label="mejor"); axs[0].plot([h[1] for h in hist], color=UNIBE["dorado"], label="media")
axs[0].set_title("Aptitud por generación"); axs[0].set_xlabel("generación"); axs[0].legend()
axs[1].bar(range(50), hist[-1][2], color=[UNIBE["naranja"] if r else UNIBE["azul"] for r in es_ruido])
axs[1].set_title("Frecuencia de cada variable en la última población (naranja = ruido)"); axs[1].set_xlabel("variable")
plt.show()
print(f"El mejor usa {mejor.sum()} variables, {int((mejor.astype(bool) & es_ruido).sum())} de ruido:")
print(", ".join(np.array(nombres)[mejor.astype(bool)]))

Comparamos en la **prueba**, que nadie usó para elegir:

In [ ]:
def en_prueba(m):
    modelo = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)).fit(X_tr[:, m], y_tr)
    return modelo.score(X_te[:, m], y_te)


for nombre, m in [("todas (50)", np.ones(50, bool)), ("RFE (10)", m_rfe), ("L1", m_l1), ("genético", mejor.astype(bool))]:
    print(f"{nombre:<12} {int(m.sum()):>2} variables · exactitud en prueba = {en_prueba(m):.3f}")

El genético encuentra un subconjunto muy pequeño, sin ruido, que en la prueba iguala a las 50 variables, explorando apenas unos cientos de subconjuntos. Su costo es el tiempo: cada aptitud entrena modelos. Y cuidado: si comparas muchos subconjuntos con la misma validación, el mejor tiene algo de **suerte** (sobreajuste de la selección); por eso la cifra honesta es la de la prueba.

## Parte 3 · Reglas difusas de Mamdani

La **lógica difusa** (Zadeh, 1965) admite grados de verdad entre 0 y 1: un servicio de 6,5 sobre 10 es «aceptable» con grado 0,7 y «bueno» con grado 0,3. Un sistema de **Mamdani** (Mamdani y Assilian, 1975) sigue cuatro pasos (A2.15):

1. **Fuzzificar** las entradas con funciones de pertenencia (aquí, triángulos).
2. **Activar** cada regla: «O» = máximo, «Y» = mínimo.
3. **Recortar** el conjunto de salida de cada regla a su activación y **unirlos** (máximo).
4. **Defuzzificar** con el **centroide**: \(\int x\,\mu(x)\,dx / \int \mu(x)\,dx\).

El ejemplo icónico es la **propina**, con las mismas reglas que la animación A2.15.

In [ ]:
def triangulo(x, a, b, c):
    """Pertenencia triangular con pie en a y c y pico en b (admite hombros: a == b o b == c)."""
    x = np.asarray(x, float)
    izq = np.where(b > a, (x - a) / (b - a + 1e-12), (x >= b).astype(float))
    der = np.where(c > b, (c - x) / (c - b + 1e-12), (x <= b).astype(float))
    return np.clip(np.minimum(izq, der), 0, 1)


SERV = {"malo": (0, 0, 5), "aceptable": (0, 5, 10), "bueno": (5, 10, 10)}
COMI = {"mala": (0, 0, 5), "aceptable": (0, 5, 10), "buena": (5, 10, 10)}
PROP = {"baja": (0, 0, 13), "media": (0, 13, 25), "alta": (13, 25, 25)}
x_prop = np.linspace(0, 25, 251)

**Tu turno.** Completa la inferencia: las activaciones de las tres reglas y el centroide.

- R1: SI la comida es mala **O** el servicio es malo ENTONCES la propina es baja.
- R2: SI el servicio es aceptable ENTONCES la propina es media.
- R3: SI el servicio es bueno **O** la comida es buena ENTONCES la propina es alta.

In [ ]:
def propina(servicio, comida):
    s = {k: float(triangulo(servicio, *v)) for k, v in SERV.items()}
    f = {k: float(triangulo(comida, *v)) for k, v in COMI.items()}
    # TODO: activaciones a1, a2, a3 y centroide de la salida agregada.
    a1 = ...
    a2 = ...
    a3 = ...
    agregada = ...
    centroide = ...
    return centroide, (a1, a2, a3), agregada


c, act, agg = propina(6.5, 9.8)
print(f"Servicio 6,5 y comida 9,8 → activaciones {np.round(act, 2)} → propina = {c:.1f} %")
assert 13 < c < 18, "La salida debería quedar entre la propina media (13 %) y la alta."

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(11, 3.6))
for nombre, (a, b, cc) in PROP.items():
    axs[0].plot(x_prop, triangulo(x_prop, a, b, cc), "--", lw=1, label=nombre)
axs[0].fill_between(x_prop, agg, alpha=0.3, color=UNIBE["azul"]); axs[0].axvline(c, color=UNIBE["verde"], lw=2.5)
axs[0].set_title(f"Salida agregada y centroide = {c:.1f} %"); axs[0].set_xlabel("propina (%)"); axs[0].legend(loc="upper center", ncol=3, fontsize=8)
grid = np.linspace(0, 10, 41)
sup = np.array([[propina(s, f)[0] for s in grid] for f in grid])
im = axs[1].imshow(sup, origin="lower", extent=[0, 10, 0, 10], cmap="Blues", aspect="auto")
plt.colorbar(im, ax=axs[1], label="propina (%)"); axs[1].set_xlabel("servicio"); axs[1].set_ylabel("comida")
axs[1].set_title("Superficie de control: tres reglas, una función suave"); axs[1].grid(False)
plt.show()

Tres reglas que cualquiera entiende generan una función suave de dos variables. Esa es la fortaleza de los sistemas difusos (control de electrodomésticos, frenos, climatización): **conocimiento experto interpretable**. Su debilidad: las reglas y las curvas las diseña una persona; con muchos datos, un modelo aprendido suele ser más preciso.

## Parte 4 · Un sistema experto y las reglas que aprende un árbol

Un **sistema experto** (MYCIN, DENDRAL; Buchanan y Shortliffe, 1984) separa tres piezas (A2.16):

- **Base de hechos:** lo que sabemos del caso.
- **Base de reglas:** «SI condiciones ENTONCES conclusión», escritas por expertos.
- **Motor de inferencia:** con **encadenamiento hacia adelante**, dispara toda regla cuyas condiciones se cumplen y agrega su conclusión, hasta que no hay nada nuevo. Además **explica** por qué concluyó lo que concluyó.

Nuestro experto diagnostica modelos de ML con lo aprendido en la sesión 1: estas son las mismas reglas de la animación.

In [ ]:
REGLAS = [
    ("R1", {"error de entrenamiento alto", "error de validación alto"}, "diagnóstico: sesgo (subajuste)"),
    ("R2", {"error de entrenamiento bajo", "brecha grande"}, "diagnóstico: varianza (sobreajuste)"),
    ("R3", {"exactitud casi perfecta", "problema difícil"}, "sospecha de fuga de datos"),
    ("R4", {"diagnóstico: sesgo (subajuste)"}, "recomendar: modelo más flexible o mejores variables"),
    ("R5", {"diagnóstico: varianza (sobreajuste)", "pocos datos"}, "recomendar: conseguir más datos"),
    ("R6", {"diagnóstico: varianza (sobreajuste)"}, "recomendar: regularizar o simplificar"),
    ("R7", {"sospecha de fuga de datos"}, "recomendar: revisar variables del futuro y el pipeline"),
    ("R8", {"clases desbalanceadas", "se reporta solo la exactitud"}, "recomendar: usar recall, precisión y curva ROC"),
]

**Tu turno.** Escribe el motor: mientras alguna regla tenga todas sus condiciones en `hechos` y su conclusión no esté todavía, agrégala y anota en `por_que` qué regla la dedujo.

In [ ]:
def inferir(hechos_iniciales):
    hechos = set(hechos_iniciales)
    por_que = {h: "hecho inicial" for h in hechos}
    # TODO: encadenamiento hacia adelante.
    ...
    return hechos, por_que


h, pq = inferir({"error de entrenamiento bajo", "brecha grande", "pocos datos"})
for hecho in sorted(h - {"error de entrenamiento bajo", "brecha grande", "pocos datos"}):
    print(f"• {hecho}\n    porque {pq[hecho]}")
assert "recomendar: conseguir más datos" in h

### El experto revisa modelos de verdad

Los hechos no tienen por qué escribirse a mano: los calculamos con métricas reales. Entrenamos dos árboles en el **Titanic**, uno sin límite de profundidad y otro de profundidad 3, y dejamos que el experto los diagnostique.

In [ ]:
from sklearn.model_selection import cross_validate
from sklearn.tree import DecisionTreeClassifier, export_text

ti = cargar_titanic()
ti["mujer"] = (ti["sex"] == "female").astype(int)
cols = ["pclass", "mujer", "age", "sibsp", "parch", "fare"]
X_ti, y_ti = ti[cols].fillna(ti["age"].median()), ti["survived"]


def hechos_de(modelo, X, y):
    """Convierte las métricas de validación cruzada en hechos para el sistema experto."""
    r = cross_validate(modelo, X, y, cv=CV, return_train_score=True)
    e_ent, e_val = 1 - r["train_score"].mean(), 1 - r["test_score"].mean()
    hechos = set()
    hechos.add("error de entrenamiento bajo" if e_ent < 0.1 else "error de entrenamiento alto" if e_ent > 0.25 else "error de entrenamiento medio")
    if e_val > 0.25:
        hechos.add("error de validación alto")
    if e_val - e_ent > 0.08:
        hechos.add("brecha grande")
    if len(y) < 1000:
        hechos.add("pocos datos")
    return hechos, e_ent, e_val


for nombre, m in [("árbol sin límite", DecisionTreeClassifier(random_state=SEED)),
                  ("árbol de profundidad 3", DecisionTreeClassifier(max_depth=3, random_state=SEED))]:
    hechos, e_ent, e_val = hechos_de(m, X_ti, y_ti)
    concl, _ = inferir(hechos)
    print(f"{nombre}: error entrenamiento {e_ent:.2f} · validación {e_val:.2f}")
    print("   conclusiones:", sorted(concl - hechos) or "ninguna regla se activa")

Ahora el contraste: el conocimiento del experto lo **escribe una persona**; el de un árbol de decisión se **aprende de los datos**, y también se puede leer como reglas SI–ENTONCES.

In [ ]:
arbol = DecisionTreeClassifier(max_depth=3, random_state=SEED).fit(X_ti, y_ti)
print(export_text(arbol, feature_names=cols, class_names=["murió", "sobrevivió"]))

Las reglas del árbol cuentan una historia que los historiadores reconocen: «mujeres y niños primero», y la clase del pasaje importó. Es **conocimiento extraído de los datos**: legible, revisable y discutible, que es lo que pide el temario.

## Parte 5 · Selección del modelo con validación cruzada anidada

Volvemos al cáncer de mama (solo las 30 variables reales). Queremos elegir entre cuatro familias **y** sus hiperparámetros sin engañarnos. Si usamos la misma validación para ajustar los hiperparámetros y para reportar el resultado, la cifra sale optimista. La **validación cruzada anidada** separa los dos trabajos:

- **Bucle interno** (`GridSearchCV`): elige hiperparámetros.
- **Bucle externo** (`cross_val_score` sobre el `GridSearchCV`): estima cuánto rinde ese procedimiento completo.

Los candidatos incluyen los dos ensambles de A2.9 y A2.10: el **bosque aleatorio** (muchos árboles independientes que votan) y el **gradient boosting** (árboles que corrigen los errores de los anteriores).

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.svm import SVC

Xr_tr, Xr_te = X_tr[:, :30], X_te[:, :30]
candidatos = {
    "logística": (make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)), {"logisticregression__C": [0.01, 0.1, 1, 10]}),
    "SVM RBF": (make_pipeline(StandardScaler(), SVC(probability=True, random_state=SEED)), {"svc__C": [0.3, 1, 3, 10], "svc__gamma": ["scale", 0.01]}),
    "bosque aleatorio": (RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=-1), {"max_features": ["sqrt", 0.3]}),
    "gradient boosting": (HistGradientBoostingClassifier(random_state=SEED), {"learning_rate": [0.05, 0.1], "max_depth": [2, None]}),
}
interno = StratifiedKFold(3, shuffle=True, random_state=1)

**Tu turno.** Para cada candidato, calcula la exactitud **anidada**: `cross_val_score(GridSearchCV(modelo, rejilla, cv=interno), Xr_tr, y_tr, cv=CV)`. Guarda también `best_score_` del `GridSearchCV` ajustado una sola vez sobre todo el entrenamiento: es la cifra que muchos reportan.

In [ ]:
# TODO: llena la lista filas con modelo, exactitud anidada (media y desviación) y la cifra optimista.
filas = []
for nombre, (modelo, rejilla) in candidatos.items():
    ...
tabla = pd.DataFrame(filas).sort_values("anidada", ascending=False)
print(tabla.round(3).to_string(index=False))

`best_score_` tiende a ser optimista porque se eligió mirando esa misma validación; aquí la diferencia es pequeña, e incluso al revés en algún modelo, porque las rejillas son pequeñas. Con rejillas grandes y pocos datos ese sesgo crece, y la cifra anidada es la que se reporta.

Las diferencias entre los cuatro son del tamaño de su desviación: con 426 tumores de entrenamiento, **no hay un ganador claro**. En ese caso conviene elegir el modelo más **simple e interpretable** (aquí, la logística) y no el que ganó por décimas. Por último, revisamos la **calibración** (A2.11): cuando el modelo dice «80 % de probabilidad de maligno», ¿acierta el 80 % de las veces? El **puntaje de Brier** resume la calibración y la discriminación en un número: el error cuadrático medio de las probabilidades (más bajo es mejor).

In [ ]:
from sklearn.calibration import calibration_curve
from sklearn.metrics import brier_score_loss
from sklearn.model_selection import cross_val_predict

# probabilidades «fuera de pliegue» en el entrenamiento: 426 casos dan curvas más estables que los 143 de la prueba
fig, ax = plt.subplots(figsize=(5.5, 4.4))
ax.plot([0, 1], [0, 1], ":", color="black", label="calibración perfecta")
for nombre in ["logística", "bosque aleatorio", "gradient boosting"]:
    modelo, rejilla = candidatos[nombre]
    final = GridSearchCV(modelo, rejilla, cv=interno, n_jobs=-1)
    p = cross_val_predict(final, Xr_tr, y_tr, cv=CV, method="predict_proba")[:, 1]
    frac, media = calibration_curve(y_tr, p, n_bins=8, strategy="quantile")
    ax.plot(media, frac, "o-", label=nombre)
    acc = final.fit(Xr_tr, y_tr).score(Xr_te, y_te)
    print(f"{nombre:<18} Brier (VC) = {brier_score_loss(y_tr, p):.3f} · exactitud en prueba = {acc:.3f}")
ax.set_xlabel("probabilidad predicha de «maligno»"); ax.set_ylabel("fracción real de malignos")
ax.set_title("Curvas de calibración (VC de 5 pliegues)"); ax.legend(); plt.show()

**Preguntas para la discusión (Clase 3):**

1. Un hospital te pide «el modelo más preciso». Con la tabla anidada en la mano, ¿qué le recomiendas y por qué?
2. ¿En qué se parece y en qué se diferencia el conocimiento de un sistema experto del de un árbol de decisión?
3. El algoritmo genético y RFE eligieron variables distintas con exactitud parecida. ¿Qué te dice eso sobre interpretar «las variables importantes»?

## Desafío opcional con IA agéntica · Selección de variables y del modelo, con un agente que justifica

**Objetivo.** Pedirle a un asistente que aplique la selección de variables y la validación cruzada anidada al Titanic y verificar que no haya fuga de información.

**Prompt inicial.** Úsalo en la herramienta que prefieras (Claude Code, Codex, Gemini en Colab, ChatGPT…), con este notebook resuelto como contexto. Pide primero un plan y revisa cada paso antes de aprobarlo.

```text
Tengo el notebook resuelto L2.3 (selección de variables con filtro, RFE, L1 y un algoritmo genético; sistema experto; validación cruzada anidada). Muéstrame primero un plan y espera mi aprobación. Luego agrega una sección con el Titanic (seaborn-data): prepara variables (sexo, clase, edad imputada, tarifa, tamaño de la familia, puerto), aplica RFE y la logística con L1 dentro de pipelines, compara logística, bosque aleatorio y gradient boosting con validación cruzada anidada, y escribe tres reglas SI–ENTONCES que resuman lo aprendido por un árbol de profundidad 3.
```

**Cómo verificar el resultado**

- La imputación y el escalado están dentro del pipeline, de modo que se ajustan en cada pliegue.
- La comparación de modelos usa la cifra anidada, no best_score_.
- Las tres reglas coinciden con las ramas que imprime export_text, no con lo que el asistente «sabe» del Titanic.

**Declara el uso de IA** (como pide el sílabo): herramienta, prompts relevantes, qué verificaste tú y qué corregiste. El desafío es opcional y no se califica; lo que cuenta es que puedas explicar cada decisión.